# Demo 2 — MLflow AI Gateway: Unified Model Router

This notebook demonstrates the MLflow AI Gateway as a model router that can transparently
route the same API call to different LLM backends.

**Prerequisites (running locally against Docker services):**
```bash
cd demo2-mlflow-gateway
docker compose up -d
# Wait for mlflow-gateway to be healthy (~2 min for model pulls)
pip install -r requirements-dev.txt
```

**Gateway:** http://localhost:5000  
**Routes API:** http://localhost:5000/api/2.0/gateway/routes/

In [ ]:
import os
import math
import time
import httpx
from dotenv import load_dotenv

load_dotenv("../.env")

# When running locally, gateway is on localhost
GATEWAY_URI = os.getenv("MLFLOW_GATEWAY_URI", "http://localhost:5000")

print(f"Gateway URI: {GATEWAY_URI}")

## 1. Inspect Available Routes

The gateway exposes a REST API. Let's list all configured routes.

In [ ]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client(GATEWAY_URI)
routes = client.list_endpoints()

print(f"Found {len(routes)} route(s):")
for r in routes:
    print(f"  {r['name']:25s}  type={r['endpoint_type']}  model={r.get('model', {}).get('name', '?')}")

## 2. Side-by-Side: llama3.2 vs. mistral

The **exact same Python code** routes to different Ollama models by changing one string.
This is the core value of a model gateway — provider/model abstraction behind a unified API.

In [ ]:
questions = [
    "What is the 2024 standard deduction for a married couple filing jointly?",
    "Explain the wash sale rule in one sentence.",
    "What is the income limit for contributing to a Roth IRA as a single filer in 2024?",
]

for question in questions:
    print(f"\nQ: {question}")
    print("-" * 70)
    payload = {"messages": [{"role": "user", "content": question}]}

    for route in ["llama-chat", "mistral-chat"]:
        t0 = time.time()
        try:
            resp = client.predict(endpoint=route, inputs=payload)
            elapsed = time.time() - t0
            answer = resp["choices"][0]["message"]["content"]
            label = "🦙 llama3.2 (3B, fast)" if route == "llama-chat" else "⚡ mistral (7B, quality)"
            print(f"{label} ({elapsed:.2f}s):\n  {answer[:200]}")
        except Exception as e:
            print(f"  [{route}] ERROR: {e}")

## 3. Embeddings via Gateway

The gateway also routes embedding requests. Here we embed tax concepts and measure
semantic similarity using cosine distance.

In [ ]:
tax_concepts = [
    "standard deduction",
    "itemized deductions",
    "capital gains tax",
    "401k retirement savings",
    "Roth IRA tax-free growth",
    "wash sale rule",
]

resp = client.predict(endpoint="ollama-embeddings", inputs={"input": tax_concepts})
vecs = [item["embedding"] for item in resp["data"]]
print(f"Embedded {len(vecs)} concepts, dimension={len(vecs[0])}\n")

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x**2 for x in a)) * math.sqrt(sum(x**2 for x in b)))

# Print similarity matrix
print("Cosine similarity matrix (1.0 = identical, 0.0 = unrelated):")
header = [c[:12] for c in tax_concepts]
print(f"{'':20s}" + " ".join(f"{h:>14s}" for h in header))
for i, concept_i in enumerate(tax_concepts):
    row = [cosine(vecs[i], vecs[j]) for j in range(len(tax_concepts))]
    print(f"{concept_i[:20]:20s}" + " ".join(f"{v:14.3f}" for v in row))

## 4. Direct HTTP — OpenAI-Compatible Invocation Endpoint

The gateway exposes each route at `/gateway/<route-name>/invocations`.
Any OpenAI-compatible client can point to the gateway without code changes.

In [ ]:
for route in ["llama-chat", "mistral-chat"]:
    url = f"{GATEWAY_URI}/gateway/{route}/invocations"
    payload = {
        "messages": [
            {"role": "user", "content": "What is the 2024 SALT deduction cap? One sentence."}
        ]
    }
    r = httpx.post(url, json=payload, timeout=60)
    data = r.json()
    answer = data["choices"][0]["message"]["content"]
    print(f"[{route}] HTTP {r.status_code}: {answer}")

## 5. Latency Benchmarking

Run the same prompt 3 times against each route and compare average latency.

In [ ]:
BENCH_PROMPT = "What are the 2024 long-term capital gains tax rates? List the brackets."
N = 3

for route in ["llama-chat", "mistral-chat"]:
    times = []
    for _ in range(N):
        t0 = time.time()
        try:
            client.predict(
                endpoint=route,
                inputs={"messages": [{"role": "user", "content": BENCH_PROMPT}]}
            )
            times.append(time.time() - t0)
        except Exception as e:
            print(f"[{route}] run failed: {e}")
            break
    if times:
        avg = sum(times) / len(times)
        print(f"[{route}]  avg={avg:.2f}s  min={min(times):.2f}s  max={max(times):.2f}s  (n={len(times)})")

## Summary

| Feature | Demonstrated |
|---|---|
| Provider abstraction | Same `client.predict()` call → llama3.2 or mistral |
| Route types | Chat completions (`llama-chat`, `mistral-chat`) + embeddings |
| Raw HTTP | OpenAI-compatible `/invocations` endpoint |
| Latency comparison | 3B fast model vs 7B quality model |

**Next:** `02_rag_evaluation.ipynb` — use `mlflow.evaluate()` to compare llama3.2 vs mistral on RAG quality metrics.